# ResNet50 · Colab v2

Clasificación de paneles solares: ImageNet congelado, fine-tuning de conv5 y pesos aleatorios.

**Empezá en una sesión nueva de Colab con GPU T4 y ejecutá de arriba hacia abajo.**

1. Dejá tu nombre en `INTEGRANTE`. Para otra corrida, cambiá `CORRIDA`.
2. Cuando se pida, pegá el enlace de **una carpeta existente** donde quieras guardar todo. Lo ideal es tu carpeta `PDI-TP/resultados/<tu nombre>`: abrila en Drive y copiá la URL del navegador.
3. Autorizá tu cuenta cuando Colab lo solicite.

Este notebook **no crea carpetas en Drive** ni usa `drive.mount`. Guarda en el disco de Colab y respalda archivos por ID en la carpeta elegida. Los checkpoints se copian al terminar cada época; si falla la copia, se detiene con los archivos locales conservados. La copia puede tardar porque los modelos son grandes.

Cada archivo remoto lleva el prefijo de integrante y corrida. Las métricas, gráficos y CSV también se reúnen en un ZIP al finalizar. Los `.keras` se guardan por separado. No se borran carpetas ni resultados anteriores.

El split y los datos originales se conservan. Las referencias N06/N16 siguen siendo provisionales; se informa además la evaluación sin esos casos. Val selecciona checkpoints: no es un test independiente.


In [ ]:
# 1. Opciones: solo hace falta cambiar estos valores.
INTEGRANTE = "nombre"  # poné tu nombre en Colab (ej. "kevo"); no subirlo con tu nombre
CORRIDA = "resnet50_01"  # Cambiar para un experimento NUEVO; no usar dos sesiones con el mismo nombre.
CARPETA_DRIVE = ""  # Opcional: pegar aquí la URL de una carpeta existente. Si queda vacío, se pregunta.

SEMILLA = 42
LADO = 256
BATCH = 16
EPOCAS_CABEZA = 10
EPOCAS_FINE = 10
EPOCAS_ALEATORIO = 20
UMBRAL = 0.5
AUGMENTATION = True


In [ ]:
# 2. Preparar dependencias y código compartido. No monta Drive.
import importlib.util
import subprocess
import sys
import os
import re
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Este notebook está preparado para Google Colab. Abrilo allí con GPU T4.")
for paquete, modulo in [
    ("tensorflow>=2.20,<2.22", "tensorflow"), ("numpy", "numpy"),
    ("pandas", "pandas"), ("Pillow", "PIL"), ("matplotlib", "matplotlib"),
    ("scikit-learn", "sklearn"), ("google-api-python-client", "googleapiclient")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

# Versión conocida: evita que cambios posteriores del equipo alteren esta corrida.
REVISION = "646bd1389ee6ef185d49f8b26b773d5fd1b34eb0"
repo = Path("/content/PDI-TP-resnet50-v2")
if not repo.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if actual != REVISION:
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REVISION], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))

import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from importlib.metadata import version
from pdi_tp import entorno, clasificacion as clf

if INTEGRANTE == "nombre":
    raise ValueError("Poné tu nombre en INTEGRANTE (primera celda): se usa para separar tus resultados.")
if not re.fullmatch(r"[a-z0-9_-]+", INTEGRANTE) or not re.fullmatch(r"[a-z0-9_-]+", CORRIDA):
    raise ValueError("Usá solo minúsculas, números, guion o guion bajo en INTEGRANTE y CORRIDA.")
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("Activá GPU T4 en Entorno de ejecución > Cambiar tipo de entorno y volvé a empezar.")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
keras.utils.set_random_seed(SEMILLA)
tf.config.experimental.enable_op_determinism()
print("Python:", platform.python_version(), "TensorFlow:", tf.__version__, "GPU:", gpus)

datos_dir = repo / "data" / "grupo_5"
if not all((datos_dir / nombre).is_dir() for nombre in ("images", "labels")):
    subprocess.run([sys.executable, str(repo / "scripts/download_data.py")], check=True)

raiz_local = Path("/content/resultados_resnet50_v2") / INTEGRANTE
salida = raiz_local / CORRIDA
salida.mkdir(parents=True, exist_ok=True)
RUN = CORRIDA
rutas = entorno.Rutas(INTEGRANTE, repo, datos_dir, datos_dir / "images", datos_dir / "labels",
                      raiz_local.parent, raiz_local)
print("Guardado local:", salida)


## Elegir y comprobar el destino

Pegá el enlace de una carpeta **que ya exista**. No hace falta agregar accesos directos ni montar Drive. La copia usa la [API de Drive por ID](https://developers.google.com/workspace/drive/api/guides/manage-uploads), con comprobación del contenido. La primera copia de `destino.json` verifica escritura antes de preparar datos o entrenar.


In [ ]:
"""Guardado por ID de carpeta. No crea carpetas en Drive."""
import hashlib
import json
import re
from pathlib import Path


def id_carpeta(enlace):
    texto = enlace.strip()
    coincidencia = re.search(r'/folders/([A-Za-z0-9_-]+)', texto)
    if coincidencia:
        return coincidencia.group(1)
    if re.fullmatch(r'[A-Za-z0-9_-]{10,}', texto):
        return texto
    raise ValueError('Pegá el enlace de una carpeta de Drive (debe contener /folders/).')


class RespaldoDrive:
    def __init__(self, servicio, folder_id, prefijo, estado, media_factory):
        self.servicio = servicio
        self.folder_id = folder_id
        self.prefijo = prefijo
        self.estado = Path(estado)
        self.media_factory = media_factory
        self.datos = json.loads(self.estado.read_text()) if self.estado.exists() else {}
        identidad = {'folder_id': folder_id, 'prefijo': prefijo}
        if self.datos and self.datos['identidad'] != identidad:
            raise ValueError('Esta corrida ya tiene otro destino. Usá otro nombre de CORRIDA.')
        self.datos.setdefault('identidad', identidad)
        self.datos.setdefault('archivos', {})
        meta = servicio.files().get(fileId=folder_id, supportsAllDrives=True,
            fields='id,name,mimeType,trashed,capabilities(canAddChildren)').execute(num_retries=3)
        if meta.get('trashed') or meta['mimeType'] != 'application/vnd.google-apps.folder':
            raise ValueError('El enlace no corresponde a una carpeta activa.')
        if not meta.get('capabilities', {}).get('canAddChildren', False):
            raise PermissionError('No tenés permiso para guardar archivos en esa carpeta.')
        self.nombre = meta['name']
        self._guardar()

    def _guardar(self):
        self.estado.parent.mkdir(parents=True, exist_ok=True)
        temporal = self.estado.with_suffix('.tmp')
        temporal.write_text(json.dumps(self.datos, indent=2), encoding='utf-8')
        temporal.replace(self.estado)

    def _buscar(self, nombre):
        seguro = nombre.replace('\\', '\\\\').replace("'", "\\'")
        q = f"'{self.folder_id}' in parents and trashed = false and name = '{seguro}'"
        resultado = self.servicio.files().list(q=q, spaces='drive', pageSize=100,
            fields='files(id,name),nextPageToken', supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute(num_retries=3)
        if resultado.get('nextPageToken') or len(resultado.get('files', [])) > 1:
            raise RuntimeError(f'Hay archivos duplicados llamados {nombre}. No se crea otro.')
        return next(iter(resultado.get('files', [])), None)

    def subir(self, ruta, relativo):
        ruta = Path(ruta)
        nombre = self.prefijo + '__' + relativo.replace('/', '__')
        with ruta.open('rb') as archivo:
            digest = hashlib.file_digest(archivo, 'md5').hexdigest()
        registro = self.datos['archivos'].get(nombre)
        if registro and registro.get('md5') == digest:
            return False
        if not registro:
            existente = self._buscar(nombre)
            if existente:
                archivo_id = existente['id']
            else:
                archivo_id = self.servicio.files().generateIds(count=1, space='drive').execute(num_retries=3)['ids'][0]
            registro = {'id': archivo_id}
            self.datos['archivos'][nombre] = registro
            # Persistir el ID ANTES de crear: un reintento usa el mismo ID.
            self._guardar()
        archivo_id = registro['id']
        from googleapiclient.errors import HttpError
        try:
            actual = self.servicio.files().get(fileId=archivo_id, supportsAllDrives=True,
                fields='id,parents,trashed,md5Checksum').execute(num_retries=3)
        except HttpError as exc:
            if exc.resp.status != 404:
                raise
            actual = None
        if actual and (actual.get('trashed') or self.folder_id not in actual.get('parents', [])):
            raise RuntimeError('Un archivo de esta corrida se movió o eliminó. Revisá Drive antes de seguir.')
        if actual and actual.get('md5Checksum') == digest:
            registro['md5'] = digest
            self._guardar()
            return False
        media = self.media_factory(str(ruta), mimetype='application/octet-stream',
            chunksize=8 * 1024 * 1024, resumable=True)
        if actual:
            request = self.servicio.files().update(fileId=archivo_id, media_body=media,
                supportsAllDrives=True, fields='id,md5Checksum')
        else:
            request = self.servicio.files().create(body={'id': archivo_id,
                'name': nombre, 'parents': [self.folder_id]}, media_body=media,
                supportsAllDrives=True, fields='id,md5Checksum')
        respuesta = None
        while respuesta is None:
            _, respuesta = request.next_chunk(num_retries=3)
        if respuesta.get('md5Checksum') != digest:
            raise IOError(f'No se pudo verificar la copia de {nombre}. El original sigue en Colab.')
        registro['md5'] = digest
        self._guardar()
        return True

    def sincronizar(self, carpeta):
        carpeta = Path(carpeta)
        cantidad = 0
        for ruta in sorted(carpeta.rglob('*')):
            if ruta.is_file() and not ruta.name.endswith('.tmp'):
                cantidad += self.subir(ruta, ruta.relative_to(carpeta).as_posix())
        print(f'Drive: {cantidad} archivo(s) actualizado(s).')


In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

auth.authenticate_user()
if not CARPETA_DRIVE.strip():
    CARPETA_DRIVE = input("Pegá la URL de la carpeta existente para tus resultados: ").strip()
folder_id = id_carpeta(CARPETA_DRIVE)
credenciales, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
servicio_drive = build("drive", "v3", credentials=credenciales, cache_discovery=False)
respaldo = RespaldoDrive(servicio_drive, folder_id, f"{INTEGRANTE}__{CORRIDA}",
    raiz_local / f".{CORRIDA}_drive.json", MediaFileUpload)

inicio = dict(integrante=INTEGRANTE, corrida=CORRIDA, carpeta_id=folder_id,
    semilla=SEMILLA, lado=LADO, batch=BATCH, augmentation=AUGMENTATION, umbral=UMBRAL,
    epocas=[EPOCAS_CABEZA, EPOCAS_FINE, EPOCAS_ALEATORIO], revision=REVISION)
marca = salida / "destino.json"
if marca.exists():
    if json.loads(marca.read_text()) != inicio:
        raise ValueError("Esta corrida ya tiene otra configuración. Cambiá CORRIDA para conservar la anterior.")
elif respaldo._buscar(f"{INTEGRANTE}__{CORRIDA}__destino.json"):
    raise RuntimeError("Esta corrida ya existe en Drive, pero no en esta sesión. Elegí otro nombre de CORRIDA para no sobrescribirla.")
marca.write_text(json.dumps(inicio, indent=2), encoding="utf-8")
respaldo.subir(marca, "destino.json")
URL_RESULTADOS = f"https://drive.google.com/drive/folders/{folder_id}"
print("Destino verificado:", respaldo.nombre, URL_RESULTADOS)
print("Se guardarán archivos aquí; no se crearán carpetas en Drive.")


## Referencia y preparación reproducible

Se auditan los 900 pares. Solo se permite excluir Z01/línea 1, de ancho cero, según criterios v1.
Se guardan un manifiesto binario y metadatos por caja, con clase y confianza originales.
No se necesita copiar imágenes para clasificación. Las cajas positivas pequeñas y las clases 1/2 se conservan.

**N06 y N16 siguen pendientes**: `y` permanece vacío y `y_original=0` conserva lo entregado.
Se guardan predicciones para las 162 imágenes. Las métricas completas y la selección de checkpoints
usan provisionalmente la referencia original; se añade análisis de sensibilidad sobre los 160 casos
sin duda registrada. Esto no confirma semánticamente las otras etiquetas ni reemplaza la segunda revisión.

In [ ]:
salida.mkdir(parents=True, exist_ok=True)
tabla, cajas = clf.preparar_tabla(rutas)
tabla.to_csv(salida / "manifiesto.csv", index=False)
cajas.to_csv(salida / "cajas_derivadas.csv", index=False)
train = tabla.query("split == 'train'").reset_index(drop=True)
val = tabla.query("split == 'val'").reset_index(drop=True)
PESOS = clf.pesos_clase(train.y.to_numpy(dtype=int))  # calculados únicamente con train
HUELLA = clf.huella_tabla(tabla)
display(tabla.groupby(["split", "y_original", "pendiente_revision"]).size().rename("imagenes").to_frame())
display(val.loc[val.pendiente_revision, ["archivo", "y", "y_original"]])
print("Pesos de clase:", PESOS, "\nHuella:", HUELLA)
config = dict(modelo="ResNet50", lado=LADO, batch=BATCH, semilla=SEMILLA,
    augmentation=AUGMENTATION, aumentos="flip horizontal/vertical + rotacion 0.25 vueltas",
    dropout=0.3, cabeza="GlobalAveragePooling2D, Dropout, Dense sigmoid",
    umbral=UMBRAL, clase_positiva=1, promedio="binary", pesos_clase=PESOS,
    criterio_datos="v1; excluir solo Z01 linea 1; conservar cajas pequenas y confianza",
    huella_manifiesto=HUELLA, n_train=len(train), n_val=len(val),
    evaluacion="provisional; referencia original; N06/N16 pendientes",
    monitor="val_loss", patience=5, optimizador="Adam", loss="binary_crossentropy",
    preprocesamiento="ResNet preprocess_input sobre RGB 0..255, incluido en modelo",
    epocas_cabeza=EPOCAS_CABEZA, epocas_fine=EPOCAS_FINE, epocas_aleatorio=EPOCAS_ALEATORIO)
clf.guardar_json(salida / "config.json", config)
clf.guardar_json(salida / "entorno.json", dict(python=platform.python_version(),
    paquetes={p: version(p) for p in ["tensorflow", "keras", "numpy", "pandas", "Pillow", "scikit-learn"]},
    gpu=[str(g) for g in gpus], detalle_gpu=[tf.config.experimental.get_device_details(g) for g in gpus]))
respaldo.sincronizar(salida)


## Modelo y checkpoints

El entrenamiento y las escrituras de Keras se hacen en disco local. La copia a Drive se realiza después de los checkpoints y del historial de cada época. Volver a ejecutar una etapa finalizada carga su mejor checkpoint; una etapa incompleta se detiene para no pisar los archivos. Si una sesión se pierde, los archivos ya confirmados quedan en Drive; una corrida nueva requiere otro nombre.


In [ ]:
"""Entrenamiento ResNet50 con TensorFlow/Keras; importar en el entorno de modelos."""
import numpy as np
import tensorflow as tf
from tensorflow import keras


def dataset(tabla, lado=256, batch=32, entrenamiento=False, semilla=42):
    """RGB 0..255 -> preprocess_input de ResNet; val conserva el orden del manifiesto."""
    ds = tf.data.Dataset.from_tensor_slices((tabla.ruta.to_list(), tabla.y_original.to_numpy(dtype="float32")))
    if entrenamiento:
        ds = ds.shuffle(len(tabla), seed=semilla, reshuffle_each_iteration=True)

    def leer(ruta, etiqueta):
        imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
        imagen = tf.image.resize(imagen, (lado, lado))
        return imagen, tf.reshape(etiqueta, (1,))

    return ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)


def construir(lado=256, preentrenado=True, augmentation=True, semilla=42):
    base = keras.applications.ResNet50(include_top=False,
        weights="imagenet" if preentrenado else None, input_shape=(lado, lado, 3))
    base.trainable = not preentrenado
    entrada = keras.Input((lado, lado, 3), name="rgb_0_255")
    x = entrada
    if augmentation:
        x = keras.Sequential([
            keras.layers.RandomFlip("horizontal_and_vertical", seed=semilla),
            keras.layers.RandomRotation(0.25, seed=semilla + 1),
        ], name="augmentation")(x)
    # ResNet espera BGR centrado en medias ImageNet. No dividir por 255.
    x = keras.applications.resnet.preprocess_input(x)
    # ImageNet: BatchNorm en inferencia incluso durante fine-tuning.
    # Desde cero: BatchNorm debe aprender sus estadísticas.
    x = base(x, training=False) if preentrenado else base(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3, seed=semilla + 2)(x)
    salida = keras.layers.Dense(1, activation="sigmoid", name="con_paneles")(x)
    return keras.Model(entrada, salida, name="clasificador_resnet50"), base


def descongelar(base):
    """Ajusta el último bloque residual conv5; mantiene BatchNormalization congelada."""
    base.trainable = True
    for capa in base.layers:
        capa.trainable = capa.name.startswith("conv5_") and not isinstance(capa, keras.layers.BatchNormalization)


def compilar(modelo, lr):
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss="binary_crossentropy", metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])


class CopiarADrive(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        respaldo.sincronizar(salida)


def entrenar(modelo, train_ds, val_ds, carpeta, epocas, lr, pesos):
    carpeta.mkdir(parents=True, exist_ok=True)
    terminado = carpeta / "completado.json"
    if terminado.exists():
        modelo.load_weights(carpeta / "mejor.keras")
        print("Etapa ya terminada; se cargó el mejor checkpoint:", carpeta.name)
        return
    if (carpeta / "historia.csv").exists() or (carpeta / "ultimo.keras").exists():
        raise RuntimeError("Esta etapa quedó incompleta. Los archivos están conservados. Para empezar otra corrida, cambiá CORRIDA en la primera celda y ejecutá desde arriba.")
    compilar(modelo, lr)
    historia = modelo.fit(train_ds, validation_data=val_ds, epochs=epocas,
        class_weight=pesos, callbacks=[
            keras.callbacks.TerminateOnNaN(),
            keras.callbacks.ModelCheckpoint(str(carpeta / "mejor.keras"), monitor="val_loss", save_best_only=True),
            keras.callbacks.ModelCheckpoint(str(carpeta / "ultimo.keras")),
            keras.callbacks.CSVLogger(str(carpeta / "historia.csv")),
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
            CopiarADrive(),
        ], verbose=2)
    if not all(np.isfinite(v).all() for v in historia.history.values()):
        raise RuntimeError("Entrenamiento no finito. No se registra como completado.")
    modelo.load_weights(carpeta / "mejor.keras")
    terminado.write_text(json.dumps({"epocas": len(historia.epoch), "monitor": "val_loss"}))
    respaldo.sincronizar(salida)
    return historia


from types import SimpleNamespace
rn = SimpleNamespace(dataset=dataset, construir=construir, descongelar=descongelar, entrenar=entrenar)


## Evaluación común y baseline

La matriz usa filas = referencia, columnas = predicción, orden `[sin paneles, con paneles]`.
Se exporta una fila por imagen con nombre, fuente, grupo, referencia, probabilidad y predicción
para la futura comparación con los detectores. Se informa también el baseline «siempre con paneles».
Los archivos `.keras` incluyen preprocesamiento y esperan imágenes RGB en 0..255.

In [ ]:
resumen = []

def evaluar_guardar(nombre, probabilidades, params, modelo=None):
    carpeta = salida / nombre
    carpeta.mkdir(parents=True, exist_ok=True)
    p = np.asarray(probabilidades).reshape(-1)
    metricas = clf.evaluar(val.y_original.to_numpy(dtype=int), p, UMBRAL)
    mascara = ~val.pendiente_revision.to_numpy()
    metricas["sin_pendientes"] = clf.evaluar(val.loc[mascara, "y_original"].to_numpy(dtype=int), p[mascara], UMBRAL)
    pred = val[["archivo", "fuente", "grupo", "y", "y_original", "pendiente_revision"]].copy()
    pred["probabilidad"] = p
    pred["prediccion"] = (p >= UMBRAL).astype(int)
    pred.to_csv(carpeta / "predicciones_val.csv", index=False)
    por_fuente = {f: clf.evaluar(g.y_original.to_numpy(dtype=int), g.probabilidad, UMBRAL)
                  for f, g in pred.groupby("fuente")}
    metricas["por_fuente"] = por_fuente
    clf.guardar_json(carpeta / "metricas.json", metricas)
    parametros = dict(config, **params, run=RUN, carpeta=str(carpeta))
    clf.guardar_json(carpeta / "registro.json", dict(nombre=f"resnet50_{RUN}_{nombre}",
        integrante=INTEGRANTE, params=parametros, metricas=metricas))
    resumen[:] = [fila for fila in resumen if fila["experimento"] != nombre]
    resumen.append(dict(experimento=nombre, **{k: metricas[k] for k in
                   ["accuracy", "precision", "recall", "f1", "f1_macro"]},
                   f1_sin_pendientes=metricas["sin_pendientes"]["f1"]))
    pd.DataFrame(resumen).to_csv(salida / "comparacion.csv", index=False)
    from sklearn.metrics import ConfusionMatrixDisplay
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(np.array(metricas["matriz_confusion"]),
        display_labels=["sin paneles", "con paneles"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"{nombre} · referencia provisional")
    fig.tight_layout()
    fig.savefig(carpeta / "matriz_confusion.png", dpi=130)
    plt.show()
    plt.close(fig)
    if modelo is not None:
        clf.guardar_json(carpeta / "parametros_modelo.json", dict(total=modelo.count_params(),
            entrenables=int(sum(np.prod(w.shape) for w in modelo.trainable_weights))))
    respaldo.sincronizar(salida)
    return pred

def curvas(nombre):
    historia = pd.read_csv(salida / nombre / "historia.csv")
    fig, axs = plt.subplots(1, 2, figsize=(10, 3))
    for ax, metrica in zip(axs, ["loss", "accuracy"]):
        ax.plot(historia.epoch + 1, historia[metrica], label="train")
        ax.plot(historia.epoch + 1, historia["val_" + metrica], label="val")
        ax.set(xlabel="Época", ylabel=metrica, title=nombre)
        ax.legend()
    fig.tight_layout()
    fig.savefig(salida / nombre / "curvas.png", dpi=130)
    plt.show()
    plt.close(fig)

evaluar_guardar("baseline_siempre_panel", np.ones(len(val)), dict(estrategia="baseline"))

## 1. ImageNet: entrenar solo la cabeza

El extractor queda congelado. `preprocess_input` convierte RGB a BGR y resta las medias de ImageNet;
no dividir los píxeles por 255. Los aumentos solo se activan al entrenar.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEMILLA)
train_ds = rn.dataset(train, LADO, BATCH, entrenamiento=True, semilla=SEMILLA)
val_ds = rn.dataset(val, LADO, BATCH)
modelo, base = rn.construir(LADO, preentrenado=True, augmentation=AUGMENTATION, semilla=SEMILLA)
modelo.summary()
rn.entrenar(modelo, train_ds, val_ds, salida / "imagenet_cabeza", EPOCAS_CABEZA, 1e-3, PESOS)
evaluar_guardar("imagenet_cabeza", modelo.predict(val_ds, verbose=0),
    dict(estrategia="cabeza", inicializacion="imagenet", lr=1e-3, epocas_max=EPOCAS_CABEZA), modelo)
curvas("imagenet_cabeza")
respaldo.sincronizar(salida)


## 2. ImageNet: ajuste fino del bloque conv5

Continúa desde el mejor checkpoint de la cabeza, que ya fue restaurado. Se descongela conv5
excepto BatchNorm, se recompila con Adam nuevo y LR 1e-5. La base se llama con `training=False`.
Se registra la dependencia del experimento anterior y las épocas adicionales.

In [ ]:
rn.descongelar(base)
rn.entrenar(modelo, train_ds, val_ds, salida / "imagenet_finetuning", EPOCAS_FINE, 1e-5, PESOS)
evaluar_guardar("imagenet_finetuning", modelo.predict(val_ds, verbose=0),
    dict(estrategia="conv5_sin_batchnorm", inicializacion="mejor_imagenet_cabeza",
         checkpoint_inicial=str(salida / "imagenet_cabeza" / "mejor.keras"),
         lr=1e-5, epocas_adicionales_max=EPOCAS_FINE), modelo)
curvas("imagenet_finetuning")
del modelo, base, train_ds, val_ds
respaldo.sincronizar(salida)


## 3. ResNet50 desde cero

Se crea una red nueva con `weights=None`, sin reutilizar pesos anteriores. Toda la red se entrena;
BatchNorm actualiza sus estadísticas. Se mantienen cabeza, aumentos y preprocesamiento para comparar.
Con 738 imágenes puede sobreajustar y necesitar mayor presupuesto; interpretar también las curvas.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEMILLA)
train_ds = rn.dataset(train, LADO, BATCH, entrenamiento=True, semilla=SEMILLA)
val_ds = rn.dataset(val, LADO, BATCH)
modelo, base = rn.construir(LADO, preentrenado=False, augmentation=AUGMENTATION, semilla=SEMILLA)
rn.entrenar(modelo, train_ds, val_ds, salida / "aleatorio", EPOCAS_ALEATORIO, 1e-4, PESOS)
predicciones = evaluar_guardar("aleatorio", modelo.predict(val_ds, verbose=0),
    dict(estrategia="toda_la_red", inicializacion="aleatoria", lr=1e-4, epocas_max=EPOCAS_ALEATORIO), modelo)
curvas("aleatorio")
respaldo.sincronizar(salida)


## Comparar e interpretar los resultados reales

La tabla se completa únicamente al ejecutar los entrenamientos. No hay métricas simuladas.
Comparar F1 por clase y F1 macro con el baseline, no solo accuracy. Revisar si los errores
se concentran por fuente, en cajas pequeñas o en anotaciones inciertas. La mejora con ajuste fino
debe acompañarse de curvas y presupuesto total; una corrida con semilla fija no mide variabilidad.
Falta repetir con otra arquitectura para completar el requisito global del ítem 2.

In [ ]:
display(pd.DataFrame(resumen).set_index("experimento"))
print("Resultados y checkpoints:", salida)
# Todas las predicciones mantienen los nombres para comparar con YOLO sobre el mismo val.
display(predicciones.query("prediccion != y_original").head(20))
import zipfile
archivo_zip = salida / "metricas_y_graficos.zip"
with zipfile.ZipFile(archivo_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for archivo in sorted(salida.rglob("*")):
        if archivo.is_file() and archivo != archivo_zip and archivo.suffix != ".keras":
            z.write(archivo, archivo.relative_to(salida))
respaldo.sincronizar(salida)
print("Corrida completa. Abrí tus resultados:", URL_RESULTADOS)
print("ZIP: métricas, CSV, configuración y gráficos. Checkpoints .keras: archivos separados.")


## Si se interrumpe una copia

Si Colab sigue activo, ejecutá la celda de abajo para reintentar la copia. Usa los mismos IDs de archivos, sin crear carpetas ni nuevas copias por nombre. No reinicies la sesión antes de rescatar los archivos locales que falten. El historial y los checkpoints de cada época confirmada ya están en Drive.

Para cargar un modelo, descargá el archivo `...__mejor.keras` y usá `keras.models.load_model(ruta)`. La entrada es RGB de 0 a 255; el preprocesamiento está dentro del modelo.


In [ ]:
# También se puede ejecutar en cualquier momento para copiar los archivos locales pendientes.
respaldo.sincronizar(salida)
print(URL_RESULTADOS)
